## tune-weights.ipynb

Finds optimal block weights for V2 and V3 without refitting a KNN model
for every combination.

### Why the naive approach is slow
The original approach rebuilt and refitted a full NearestNeighbors model
for every weight combination — that is O(n_albums × n_features) per trial
and took ~40 minutes for 36 combinations.

### Fast approach: weighted query-time dot product
After L2 normalisation, cosine similarity is just a dot product. Instead
of rebuilding the model, we:

1. Normalise each feature block **independently** once (stored as dense float32)
2. At query time, combine blocks with trial weights, re-normalise the
   **query vector only** (one row, near-instant), then compute dot products
   against the pre-stored full matrix
3. Take the top-N results

This reduces each trial from ~60s to ~0.1s — the full 36-combo V2 grid
runs in under a minute.

### Strategy
| Model | Free weights | Approach |
|---|---|---|
| V2 | `W_COUNTRY`, `W_TRACK_STATS` | Grid search 6×6 = 36 combos |
| V3 | V2 weights fixed + `W_ROLE_FAMILY`, `W_INSTRUMENT`, `W_CONTRIB_CNT` | Random search 80 samples |

In [1]:
import pickle
import itertools
import random
import time
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.sparse import csr_matrix, hstack, load_npz
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'
N_RESULTS    = 10
RANDOM_SEED  = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [3]:
# ── Ground truth ──────────────────────────────────────────────────────────
# Load the scope-filtered ground truth built in evaluate-lastfm.ipynb.
# If running standalone, rebuild it here.

try:
    # Reuse from memory if evaluate-lastfm.ipynb ran in the same kernel
    _ = ground_truth
    _ = mb_lookup
    print(f'Ground truth in memory: {len(ground_truth):,} seed albums')
except NameError:
    # Rebuild from scratch
    try:
        lastfm = pd.read_parquet(f'{DATA_DIR}/lastfm_album_similarity.parquet')
    except Exception:
        lastfm = pd.read_csv(f'{DATA_DIR}/lastfm_album_similarity.csv')

    lastfm.columns = lastfm.columns.str.strip()
    for col in ['album','artist','similar_album','similar_artist']:
        lastfm[col + '_key'] = lastfm[col].str.lower().str.strip()

    mb_lookup = (
        pd.read_parquet(f'{DATA_DIR}/mb_album_artists.parquet',
                        columns=['album_id','album_name','artist_name'])
        .drop_duplicates(subset='album_id')
    )
    mb_lookup['album_key']  = mb_lookup['album_name'].str.lower().str.strip()
    mb_lookup['artist_key'] = mb_lookup['artist_name'].str.lower().str.strip()
    mb_index = mb_lookup.set_index(['album_key','artist_key'])['album_id']

    seed_pairs = lastfm[['album_key','artist_key']].drop_duplicates()
    seed_pairs = seed_pairs.join(mb_index.rename('seed_mb_id'),
                                  on=['album_key','artist_key'], how='left')
    lastfm = lastfm.join(mb_index.rename('similar_mb_id'),
                          on=['similar_album_key','similar_artist_key'], how='left')
    lastfm = lastfm.merge(seed_pairs, on=['album_key','artist_key'], how='left')

    # Load all model ids to define studio-album universe
    ids_v1 = np.load(f'{DATA_DIR}/model/album_ids_annotated.npy',       allow_pickle=True)
    ids_v2 = np.load(f'{DATA_DIR}/model_v2/album_ids_annotated_v2.npy', allow_pickle=True)
    ids_v3 = np.load(f'{DATA_DIR}/model_v3/album_ids_annotated_v3.npy', allow_pickle=True)
    all_model_ids = set(ids_v1) | set(ids_v2) | set(ids_v3)

    gt_filtered = (
        lastfm
        .dropna(subset=['seed_mb_id','similar_mb_id'])
        .pipe(lambda df: df[df['similar_mb_id'].isin(all_model_ids)])
    )
    ground_truth = (
        gt_filtered
        .groupby('seed_mb_id')['similar_mb_id']
        .apply(set)
        .to_dict()
    )
    print(f'Ground truth rebuilt: {len(ground_truth):,} seed albums')

Ground truth rebuilt: 2,430 seed albums


In [4]:
# ── Feature blocks (loaded once, reused across all weight combinations) ───
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_id_order = pickle.load(f)

X_tags        = load_npz(f'{FEATURES_DIR}/album_tags_matrix.npz')
X_labels      = load_npz(f'{FEATURES_DIR}/album_labels_matrix.npz')
X_types       = load_npz(f'{FEATURES_DIR}/album_types_matrix.npz')
X_ratings     = load_npz(f'{FEATURES_DIR}/album_ratings_matrix.npz')
X_country     = load_npz(f'{FEATURES_DIR}/album_country_matrix.npz')
X_track_stats = load_npz(f'{FEATURES_DIR}/album_track_stats_matrix.npz')
X_role_family = load_npz(f'{FEATURES_DIR}/album_role_family_matrix.npz')
X_instrument  = load_npz(f'{FEATURES_DIR}/album_instrument_matrix.npz')
X_contrib_cnt = load_npz(f'{FEATURES_DIR}/album_contributor_counts_matrix.npz')

# Expand all blocks to full album universe (same logic as knn-v2/v3)
full_album_ids = pd.Index(
    pd.read_parquet(f'{DATA_DIR}/mb_album.parquet', columns=['id'])['id'].sort_values()
)

def _expand(X, row_pos, n):
    coo = X.tocoo()
    return csr_matrix((coo.data, (row_pos[coo.row], coo.col)), shape=(n, X.shape[1]))

if len(album_id_order) < len(full_album_ids):
    pos   = full_album_ids.get_indexer(album_id_order)
    n_full = len(full_album_ids)
    X_tags        = _expand(X_tags,        pos, n_full)
    X_labels      = _expand(X_labels,      pos, n_full)
    X_types       = _expand(X_types,       pos, n_full)
    X_ratings     = _expand(X_ratings,     pos, n_full)
    X_country     = _expand(X_country,     pos, n_full)
    X_track_stats = _expand(X_track_stats, pos, n_full)
    X_role_family = _expand(X_role_family, pos, n_full)
    X_instrument  = _expand(X_instrument,  pos, n_full)
    X_contrib_cnt = _expand(X_contrib_cnt, pos, n_full)
    album_id_order = full_album_ids.tolist()

print('Feature blocks ready.')
print(f'Full album universe: {len(album_id_order):,}')

Feature blocks ready.
Full album universe: 2,241,402


In [5]:
# ── Pre-compute per-block normalised matrices (done ONCE) ────────────────
#
# Strategy: store each block as a dense float32 array, one row per
# annotated album. At query time we weight + stack + renormalise only
# the query row, then score with a single dot product against the
# pre-stacked full matrix. No KNN refit needed per trial.

from sklearn.preprocessing import normalize as sk_normalize

# Determine the annotated (has-features) row mask from the V1-equivalent
# block combination so the index is consistent across all trials.
X_base_full = hstack([
    X_tags, X_labels, X_types, X_ratings
]).tocsr()
row_lengths  = np.diff(X_base_full.indptr)
has_feat     = row_lengths > 0
ann_ids      = np.array(album_id_order)[has_feat]
id2row_ann   = {aid: i for i, aid in enumerate(ann_ids)}
n_ann        = has_feat.sum()

print(f'Annotated albums: {n_ann:,}')
print('Pre-normalising blocks (this runs once)...')

def _to_dense_ann(X):
    """Subset to annotated rows, convert to dense float32."""
    arr = X[has_feat].toarray().astype(np.float32)
    # Per-row L2 norm — will be used for scaling at query time
    return arr

BLK = {
    'tags':        _to_dense_ann(X_tags),
    'labels':      _to_dense_ann(X_labels),
    'types':       _to_dense_ann(X_types),
    'ratings':     _to_dense_ann(X_ratings),
    'country':     _to_dense_ann(X_country),
    'track_stats': _to_dense_ann(X_track_stats),
    'role_family': _to_dense_ann(X_role_family),
    'instrument':  _to_dense_ann(X_instrument),
    'contrib_cnt': _to_dense_ann(X_contrib_cnt),
}

for k, v in BLK.items():
    print(f'  {k:<14} {v.shape}  {v.nbytes/1e6:.0f} MB')

print('\nBlock pre-computation done.')

Annotated albums: 1,008,102
Pre-normalising blocks (this runs once)...
  tags           (1008102, 3041)  12263 MB
  labels         (1008102, 3469)  13988 MB
  types          (1008102, 10)  40 MB
  ratings        (1008102, 1)  4 MB
  country        (1008102, 2263)  9125 MB
  track_stats    (1008102, 12)  48 MB
  role_family    (1008102, 7)  28 MB
  instrument     (1008102, 591)  2383 MB
  contrib_cnt    (1008102, 7)  28 MB

Block pre-computation done.


In [6]:
# ── Weighted evaluation (fast) ────────────────────────────────────────────

def build_matrix(weights: dict) -> np.ndarray:
    """
    Stack weighted blocks horizontally for all annotated albums.
    Returns an L2-normalised dense float32 matrix (n_ann × total_features).
    """
    parts = [
        BLK['tags']        * weights.get('W_TAGS',        1.0),
        BLK['labels']      * weights.get('W_LABELS',      1.0),
        BLK['types']       * weights.get('W_TYPES',       1.0),
        BLK['ratings']     * weights.get('W_RATINGS',     1.0),
        BLK['country']     * weights.get('W_COUNTRY',     0.0),
        BLK['track_stats'] * weights.get('W_TRACK_STATS', 0.0),
    ]
    if 'W_ROLE_FAMILY' in weights:
        parts += [
            BLK['role_family'] * weights['W_ROLE_FAMILY'],
            BLK['instrument']  * weights['W_INSTRUMENT'],
            BLK['contrib_cnt'] * weights['W_CONTRIB_CNT'],
        ]
    M = np.concatenate(parts, axis=1)
    # L2-normalise each row
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return (M / norms).astype(np.float32)


def evaluate_matrix(M: np.ndarray, ground_truth: dict, n: int = 10) -> dict:
    """
    Score the matrix M against ground_truth using batch dot products.
    M must already be L2-normalised (rows are unit vectors).
    """
    hits, precs, rrs = [], [], []

    for seed_id, known in ground_truth.items():
        if seed_id not in id2row_ann:
            continue
        row = id2row_ann[seed_id]
        q   = M[row]                          # query vector (already normalised)

        # Cosine similarity = dot product (all rows are unit vectors)
        sims = M.dot(q)                       # shape (n_ann,)
        sims[row] = -1.0                      # exclude self

        top_idx = np.argpartition(sims, -n * 3)[-n * 3:]   # fast approx top
        top_idx = top_idx[np.argsort(sims[top_idx])[::-1]]  # sort descending

        recs = [ann_ids[i] for i in top_idx[:n * 3]
                if ann_ids[i] != seed_id][:n]

        rec_set = set(recs)
        overlap = rec_set & known
        hits.append(1 if overlap else 0)
        precs.append(len(overlap) / n)
        rr = 0.0
        for rank, aid in enumerate(recs, 1):
            if aid in known:
                rr = 1.0 / rank
                break
        rrs.append(rr)

    return {
        'hit_rate':  float(np.mean(hits)),
        'precision': float(np.mean(precs)),
        'mrr':       float(np.mean(rrs)),
        'n_eval':    len(hits),
    }


# V1 baseline
import time
print('V1 baseline...')
t0 = time.time()
M_v1     = build_matrix({})           # country=0, track_stats=0
v1_scores = evaluate_matrix(M_v1, ground_truth, N_RESULTS)
print(f'V1  hit={v1_scores["hit_rate"]:.4f}  prec={v1_scores["precision"]:.4f}  mrr={v1_scores["mrr"]:.4f}  ({time.time()-t0:.1f}s)')

V1 baseline...


MemoryError: Unable to allocate 33.0 GiB for an array with shape (1008102, 8796) and data type float32

In [ ]:
# ── V2 grid search ────────────────────────────────────────────────────────
GRID_COUNTRY     = [0.05, 0.1, 0.2, 0.35, 0.5, 1.0]
GRID_TRACK_STATS = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5]

v2_results = []
total = len(GRID_COUNTRY) * len(GRID_TRACK_STATS)
print(f'V2 grid search: {total} combinations...')
t0 = time.time()

for i, (wc, wt) in enumerate(itertools.product(GRID_COUNTRY, GRID_TRACK_STATS), 1):
    weights = {'W_COUNTRY': wc, 'W_TRACK_STATS': wt}
    M       = build_matrix(weights)
    scores  = evaluate_matrix(M, ground_truth, N_RESULTS)
    v2_results.append({**weights, **scores})
    if i % 6 == 0:
        best = max(v2_results, key=lambda r: r['hit_rate'])
        elapsed = time.time() - t0
        eta     = elapsed / i * (total - i)
        print(f'  [{i}/{total}]  best: country={best["W_COUNTRY"]}  track={best["W_TRACK_STATS"]}  '
              f'hit={best["hit_rate"]:.4f}  elapsed={elapsed:.0f}s  eta={eta:.0f}s')

v2_df   = pd.DataFrame(v2_results).sort_values('hit_rate', ascending=False)
v2_best = v2_df.iloc[0]
W_COUNTRY_FIXED     = float(v2_best['W_COUNTRY'])
W_TRACK_STATS_FIXED = float(v2_best['W_TRACK_STATS'])

print(f'\nTotal time: {time.time()-t0:.1f}s')
print(f'\n=== V2 best ===')
print(v2_df.head(10).to_string(index=False))

In [ ]:
# Hit Rate heatmap for V2 grid
pivot = v2_df.pivot(index='W_COUNTRY', columns='W_TRACK_STATS', values='hit_rate')

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Hit Rate @10')
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels(pivot.columns)
ax.set_yticklabels(pivot.index)
ax.set_xlabel('W_TRACK_STATS')
ax.set_ylabel('W_COUNTRY')
ax.set_title('V2 Hit Rate @10 grid search')

# Annotate cells
for (r, c), val in np.ndenumerate(pivot.values):
    ax.text(c, r, f'{val:.3f}', ha='center', va='center', fontsize=8)

# Mark best cell
br = list(pivot.index).index(v2_best['W_COUNTRY'])
bc = list(pivot.columns).index(v2_best['W_TRACK_STATS'])
ax.add_patch(plt.Rectangle((bc-0.5, br-0.5), 1, 1, fill=False, edgecolor='blue', lw=2))

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/tune_v2_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── V3 random search ──────────────────────────────────────────────────────
N_SAMPLES = 80
print(f'V3 random search: {N_SAMPLES} samples  '
      f'(W_COUNTRY={W_COUNTRY_FIXED}, W_TRACK_STATS={W_TRACK_STATS_FIXED} fixed)')
t0 = time.time()

V3_SPACE = {
    'W_ROLE_FAMILY':  (0.1, 2.0),
    'W_INSTRUMENT':   (0.1, 2.0),
    'W_CONTRIB_CNT':  (0.05, 1.0),
}

def log_uniform(lo, hi):
    return float(np.exp(np.random.uniform(np.log(lo), np.log(hi))))

v3_results = []
for i in range(1, N_SAMPLES + 1):
    sample = {
        'W_COUNTRY':     W_COUNTRY_FIXED,
        'W_TRACK_STATS': W_TRACK_STATS_FIXED,
        'W_ROLE_FAMILY': log_uniform(*V3_SPACE['W_ROLE_FAMILY']),
        'W_INSTRUMENT':  log_uniform(*V3_SPACE['W_INSTRUMENT']),
        'W_CONTRIB_CNT': log_uniform(*V3_SPACE['W_CONTRIB_CNT']),
    }
    M      = build_matrix(sample)
    scores = evaluate_matrix(M, ground_truth, N_RESULTS)
    v3_results.append({**sample, **scores})
    if i % 10 == 0:
        best    = max(v3_results, key=lambda r: r['hit_rate'])
        elapsed = time.time() - t0
        eta     = elapsed / i * (N_SAMPLES - i)
        print(f'  [{i}/{N_SAMPLES}]  best hit={best["hit_rate"]:.4f}  '
              f'rf={best["W_ROLE_FAMILY"]:.3f}  instr={best["W_INSTRUMENT"]:.3f}  '
              f'cnt={best["W_CONTRIB_CNT"]:.3f}  elapsed={elapsed:.0f}s  eta={eta:.0f}s')

v3_df   = pd.DataFrame(v3_results).sort_values('hit_rate', ascending=False)
v3_best = v3_df.iloc[0]

print(f'\nTotal time: {time.time()-t0:.1f}s')
print(f'\n=== V3 best ===')
print(v3_df.head(10).round(4).to_string(index=False))

In [ ]:
# ── Final comparison ──────────────────────────────────────────────────────
M_v2 = build_matrix({'W_COUNTRY': W_COUNTRY_FIXED, 'W_TRACK_STATS': W_TRACK_STATS_FIXED})
v2_final = evaluate_matrix(M_v2, ground_truth, N_RESULTS)

M_v3 = build_matrix(dict(v3_best[[
    'W_COUNTRY','W_TRACK_STATS','W_ROLE_FAMILY','W_INSTRUMENT','W_CONTRIB_CNT'
]]))
v3_final = evaluate_matrix(M_v3, ground_truth, N_RESULTS)

print('=== Final comparison ===')
print(f'{"Model":<12} {"Hit Rate":>10} {"Precision":>10} {"MRR":>8}')
print('-' * 44)
print(f'{"V1 baseline":<12} {v1_scores["hit_rate"]:>10.4f} {v1_scores["precision"]:>10.4f} {v1_scores["mrr"]:>8.4f}')
print(f'{"V2 tuned":<12} {v2_final["hit_rate"]:>10.4f} {v2_final["precision"]:>10.4f} {v2_final["mrr"]:>8.4f}')
print(f'  country={W_COUNTRY_FIXED}  track_stats={W_TRACK_STATS_FIXED}')
print(f'{"V3 tuned":<12} {v3_final["hit_rate"]:>10.4f} {v3_final["precision"]:>10.4f} {v3_final["mrr"]:>8.4f}')
print(f'  rf={v3_best["W_ROLE_FAMILY"]:.3f}  instr={v3_best["W_INSTRUMENT"]:.3f}  cnt={v3_best["W_CONTRIB_CNT"]:.3f}')

In [ ]:
# ── Save best weights to disk for use in knn-v2.ipynb / knn-v3.ipynb ─────
import json

best_weights = {
    'v2': {
        'W_TAGS':        1.0,
        'W_LABELS':      1.0,
        'W_TYPES':       1.0,
        'W_RATINGS':     1.0,
        'W_COUNTRY':     W_COUNTRY_FIXED,
        'W_TRACK_STATS': W_TRACK_STATS_FIXED,
    },
    'v3': {
        'W_TAGS':        1.0,
        'W_LABELS':      1.0,
        'W_TYPES':       1.0,
        'W_RATINGS':     1.0,
        'W_COUNTRY':     float(v3_best['W_COUNTRY']),
        'W_TRACK_STATS': float(v3_best['W_TRACK_STATS']),
        'W_ROLE_FAMILY': float(v3_best['W_ROLE_FAMILY']),
        'W_INSTRUMENT':  float(v3_best['W_INSTRUMENT']),
        'W_CONTRIB_CNT': float(v3_best['W_CONTRIB_CNT']),
    }
}

out_path = f'{DATA_DIR}/best_weights.json'
with open(out_path, 'w') as f:
    json.dump(best_weights, f, indent=2)

print(f'Best weights saved to {out_path}')
print(json.dumps(best_weights, indent=2))

In [ ]:
# ── V3 random search scatter — shows which weights matter most ────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('V3 random search: Hit Rate @10 vs each weight', fontsize=12)

for ax, col in zip(axes, ['W_ROLE_FAMILY', 'W_INSTRUMENT', 'W_CONTRIB_CNT']):
    ax.scatter(v3_df[col], v3_df['hit_rate'], alpha=0.5, s=25, color='#C8522A')
    ax.axvline(float(v3_best[col]), color='blue', linestyle='--', linewidth=1.2,
               label=f'best={v3_best[col]:.3f}')
    ax.set_xlabel(col, fontsize=10)
    ax.set_ylabel('Hit Rate @10' if ax == axes[0] else '')
    ax.set_xscale('log')
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/tune_v3_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── V3 focused search: narrow ranges informed by scatter plots ────────────
#
# Observations from tune_v3_scatter.png:
#   W_INSTRUMENT  : sharp decline above ~0.15 → search 0.01–0.15 (below prev min)
#   W_ROLE_FAMILY : decline above ~0.3       → search 0.05–0.35
#   W_CONTRIB_CNT : no trend at all           → drop it (set to 0.0)
#
# Use a denser grid for the two meaningful weights now that ranges are tight.

GRID_RF    = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35]
GRID_INSTR = [0.01, 0.03, 0.05, 0.08, 0.10, 0.12, 0.15]

focused_results = []
total = len(GRID_RF) * len(GRID_INSTR)
print(f'Focused V3 grid: {total} combinations  (W_CONTRIB_CNT=0.0)')
t0 = time.time()

for i, (wrf, wi) in enumerate(
        itertools.product(GRID_RF, GRID_INSTR), 1):
    weights = {
        'W_COUNTRY':     W_COUNTRY_FIXED,
        'W_TRACK_STATS': W_TRACK_STATS_FIXED,
        'W_ROLE_FAMILY': wrf,
        'W_INSTRUMENT':  wi,
        'W_CONTRIB_CNT': 0.0,
    }
    M      = build_matrix(weights)
    scores = evaluate_matrix(M, ground_truth, N_RESULTS)
    focused_results.append({**weights, **scores})

focused_df   = pd.DataFrame(focused_results).sort_values('hit_rate', ascending=False)
focused_best = focused_df.iloc[0]

print(f'Total time: {time.time()-t0:.1f}s')
print(f'\n=== Focused V3 best ===')
print(focused_df.head(10).round(4).to_string(index=False))

# Compare against V2 and V1
print(f'\n=== Updated comparison ===')
print(f'{"Model":<14} {"Hit Rate":>10} {"Precision":>10} {"MRR":>8}')
print('-' * 46)
print(f'{"V1 baseline":<14} {v1_scores["hit_rate"]:>10.4f} '
      f'{v1_scores["precision"]:>10.4f} {v1_scores["mrr"]:>8.4f}')
v2_final = evaluate_matrix(
    build_matrix({'W_COUNTRY': W_COUNTRY_FIXED, 'W_TRACK_STATS': W_TRACK_STATS_FIXED}),
    ground_truth, N_RESULTS
)
print(f'{"V2 tuned":<14} {v2_final["hit_rate"]:>10.4f} '
      f'{v2_final["precision"]:>10.4f} {v2_final["mrr"]:>8.4f}')
focused_scores = evaluate_matrix(
    build_matrix(dict(focused_best[[
        'W_COUNTRY','W_TRACK_STATS','W_ROLE_FAMILY','W_INSTRUMENT','W_CONTRIB_CNT'
    ]])),
    ground_truth, N_RESULTS
)
print(f'{"V3 focused":<14} {focused_scores["hit_rate"]:>10.4f} '
      f'{focused_scores["precision"]:>10.4f} {focused_scores["mrr"]:>8.4f}')
print(f'  rf={focused_best["W_ROLE_FAMILY"]:.3f}  '
      f'instr={focused_best["W_INSTRUMENT"]:.3f}  cnt=0.0')

In [ ]:
# Heatmap: W_ROLE_FAMILY × W_INSTRUMENT
pivot = focused_df.pivot(
    index='W_ROLE_FAMILY', columns='W_INSTRUMENT', values='hit_rate'
)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Hit Rate @10')
ax.set_xticks(range(len(pivot.columns)))
ax.set_yticks(range(len(pivot.index)))
ax.set_xticklabels([f'{v:.2f}' for v in pivot.columns])
ax.set_yticklabels([f'{v:.2f}' for v in pivot.index])
ax.set_xlabel('W_INSTRUMENT')
ax.set_ylabel('W_ROLE_FAMILY')
ax.set_title('V3 focused grid: Hit Rate @10  (W_CONTRIB_CNT=0)')

for (r, c), val in np.ndenumerate(pivot.values):
    ax.text(c, r, f'{val:.4f}', ha='center', va='center', fontsize=7)

br = list(pivot.index).index(focused_best['W_ROLE_FAMILY'])
bc = list(pivot.columns).index(focused_best['W_INSTRUMENT'])
ax.add_patch(plt.Rectangle(
    (bc-0.5, br-0.5), 1, 1, fill=False, edgecolor='blue', lw=2
))

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/tune_v3_focused_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Load tuned weights and apply to knn-v2 / knn-v3 ─────────────────────
import json

with open(f'{DATA_DIR}/best_weights.json') as f:
    best_weights = json.load(f)

# ── V2 weights ────────────────────────────────────────────────────────────
W_TAGS        = best_weights['v2']['W_TAGS']
W_LABELS      = best_weights['v2']['W_LABELS']
W_TYPES       = best_weights['v2']['W_TYPES']
W_RATINGS     = best_weights['v2']['W_RATINGS']
W_COUNTRY     = best_weights['v2']['W_COUNTRY']
W_TRACK_STATS = best_weights['v2']['W_TRACK_STATS']

print('V2 weights loaded:')
for k, v in best_weights['v2'].items():
    print(f'  {k:<18} = {v}')

# ── V3 weights (includes all V2 weights + contributor blocks) ─────────────
W_TAGS_V3        = best_weights['v3']['W_TAGS']
W_LABELS_V3      = best_weights['v3']['W_LABELS']
W_TYPES_V3       = best_weights['v3']['W_TYPES']
W_RATINGS_V3     = best_weights['v3']['W_RATINGS']
W_COUNTRY_V3     = best_weights['v3']['W_COUNTRY']
W_TRACK_STATS_V3 = best_weights['v3']['W_TRACK_STATS']
W_ROLE_FAMILY    = best_weights['v3']['W_ROLE_FAMILY']
W_INSTRUMENT     = best_weights['v3']['W_INSTRUMENT']
W_CONTRIB_CNT    = best_weights['v3']['W_CONTRIB_CNT']

print('\nV3 weights loaded:')
for k, v in best_weights['v3'].items():
    print(f'  {k:<18} = {v}')

# ── Rebuild and save both models with tuned weights ───────────────────────
import os, joblib
from scipy.sparse import hstack, save_npz
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

def build_and_save(weights, model_dir):
    os.makedirs(f'{DATA_DIR}/{model_dir}', exist_ok=True)

    blocks = [
        X_tags        * weights.get('W_TAGS',        1.0),
        X_labels      * weights.get('W_LABELS',      1.0),
        X_types       * weights.get('W_TYPES',       1.0),
        X_ratings     * weights.get('W_RATINGS',     1.0),
        X_country     * weights.get('W_COUNTRY',     0.0),
        X_track_stats * weights.get('W_TRACK_STATS', 0.0),
    ]
    if weights.get('W_ROLE_FAMILY', 0.0) > 0:
        blocks += [
            X_role_family * weights.get('W_ROLE_FAMILY', 0.0),
            X_instrument  * weights.get('W_INSTRUMENT',  0.0),
            X_contrib_cnt * weights.get('W_CONTRIB_CNT', 0.0),
        ]

    X_full      = hstack(blocks).tocsr()
    col_nnz     = np.diff(X_full.tocsc().indptr)
    row_lengths = np.diff(X_full.indptr)
    has_feat    = row_lengths > 0
    nonempty    = np.where(has_feat)[0]
    col_vals    = col_nnz[X_full.indices]
    max_col     = np.zeros(X_full.shape[0], dtype=col_nnz.dtype)
    max_col[nonempty] = np.maximum.reduceat(col_vals, X_full.indptr[nonempty])
    safe_thresh = int(max_col[has_feat].min())
    X_pruned    = X_full[:, col_nnz >= safe_thresh]

    X_ann  = X_pruned[has_feat].copy()
    ids_ann = np.array(album_id_order)[has_feat]
    np.nan_to_num(X_ann.data, nan=0.0, copy=False)
    X_ann.eliminate_zeros()
    X_norm = normalize(X_ann, norm='l2')

    model = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
    model.fit(X_norm)

    suffix = model_dir.replace('model', '').replace('_', '') or ''
    joblib.dump(model,  f'{DATA_DIR}/{model_dir}/knn_model{suffix}.joblib')
    save_npz(f'{DATA_DIR}/{model_dir}/X_knn_norm{suffix}.npz', X_norm)
    np.save(f'{DATA_DIR}/{model_dir}/album_ids_annotated{suffix}.npy', ids_ann)
    np.save(f'{DATA_DIR}/{model_dir}/has_features{suffix}.npy', has_feat)

    print(f'Saved to {model_dir}/  '
          f'({X_norm.shape[0]:,} albums x {X_norm.shape[1]:,} features)')


print('\nRebuilding V2 with tuned weights...')
build_and_save(best_weights['v2'], 'model_v2')

print('Rebuilding V3 with tuned weights...')
build_and_save(best_weights['v3'], 'model_v3')

print('\nDone. Models saved and ready for use in app_sql_v3.py')